# FDTD 1D tutorial

The objective of this tutorial is to learn the basics of how to run OMxRTA and the steps needed to simulate the electron dynamics of a molecule under electronic strong coupling regime in a driven FP-cavity. For this, we need to:
- Select a molecule.
- Calculate the absorption spectra.
- Build the cavity.
- Simulate the system cavity+molecule.

Before starting, we need to execute the following block to get some processing tools available as well to create the input files.

In [ ]:
import sys
from pathlib import Path
import numpy as np
from os.path import join
from scipy.constants import Planck as h, c, elementary_charge as e0

# Add the repository tools directory to Python path.
tools_dir = Path.cwd().resolve().parents[1] / "tools"
if str(tools_dir) not in sys.path:
    sys.path.insert(0, str(tools_dir))

import input_creators as create
import basic_data_processors as process

## Select the molecule

For this example, we will work with an ideal and simple system, which is the $N_2$ molecule. First we will create our file `molecule_0000001.in`,
where we will include all the information needed by `OMxRTA`and `dftb+`. For this we will use the function `create.dftb_molecule_file`. To know the variables we need for this file we run:

In [ ]:
create.dftb_molecule_file(help=True)

Now we indicate the variables and we create our file:

In [ ]:
n_atoms                   = 2
n_atom_types              = 1
dynamics_type             = "electrons"
print_coordinates         = False
atom_types_list           = ["N"]
max_angular_momentum_list = ["p"]
atoms_list                = ["N", "N"]
atom_coordinates          = np.array([[ 0.55384961279772837, 0.0, 0.0], 
                                      [-0.55384961279772837, 0.0, 0.0]])

create.dftb_molecule_file(file_number=1, n_atoms=n_atoms, n_atom_types=n_atom_types, dynamics_type=dynamics_type, 
                          print_coordinates=print_coordinates, atom_types_list=atom_types_list, 
                          max_angular_momentum_list=max_angular_momentum_list, 
                          atoms_list=atoms_list, atom_coordinates=atom_coordinates)

## Calculate the absorption spectra

For this, we need to run two quick simulations. First, we use a short pulse (white light) with no molecules. And then, we repeat the simulation, including our molecule.

### Empty box case

To run our code, first we need to create a work directory. We will call it `empty_box`, and will use the following command to create it

In [ ]:
! mkdir empty_box

Now we create the `inp` file, which is the input file that contains the main variables to run our simulation. For this we will use the function `create.inp_file`. To check again all the possible variables we can run:

In [ ]:
create.inp_file(help=True)

For the moment we only need the following:

In [ ]:
dim         = 1
boundaries  = ["cpml"]
box_size    = [2000.0]
total_time  = 300.0
n_detectors = 2
print_time  = 0.05
dr          = 1.0
dt          = 0.01

create.inp_file(mxll_dimensions=dim, mxll_boundaries=boundaries, mxll_box_size=box_size, 
                mxll_total_time=total_time, mxll_dr=dr, mxll_n_detectors=n_detectors, 
                mxll_dt_det_print=print_time, mxll_dt=dt)

We also need to include at list one source in the file `sources.in`. For this we use: 

In [ ]:
source    = "point"
pol       = "x"
field_amp = 1.0E-3
freq      = 14.0
tau       = 0.2
t0        = 5.0
t_init    = 0.0
t_end     = 10.0
r0        = [500.0, 0.0, 0.0]

create.sources_file(source_type=source, polarization=pol, field_amp=field_amp, 
                    frequency=freq, t0=t0, tau=tau, t_init=t_init, t_end=t_end, 
                    r0=r0, radius=0.0, phase=0.0)

We also need to indicate the position of our dectectors and the field we want to detect, in this case:

In [ ]:
detector_type = "point"
x_min         = -500.0

create.detectors_file(detector_type=detector_type, detected_field="Ex" , x_min=x_min)
create.detectors_file(detector_type=detector_type, detected_field="Hy" , x_min=x_min, file_exist=True)

Finally, we move all the input files to our work directory `empty_box`, and run our executable in that directory:

In [ ]:
! cp inp sources.in detectors.in empty_box/.

In [ ]:
! cd empty_box && ../../../build/bin/OMxRTA.e

We can get the power transmitted at the detector position, calculated as:

$P(\omega) = -\mathrm{Re}\left\{ E_x^*(\omega) H_y(\omega) \right\}$

sing another tool function, `process.plot_trans_spectrum`.

In [ ]:
process.plot_trans_spectrum_1D(directory="empty_box")

### Box with molecule

Now let's see what happens when we include the molecule.

First, we need to create another work directory. Let's call it `box_with_1mol`.

In [ ]:
! mkdir box_with_1mol

Now we create again another `inp` file with the following variables:

In [ ]:
dim          = 1
boundaries   = ["cpml", "none", "none"]
box_size     = [2000.0, 0.0, 0.0]
total_time   = 300.0
n_detectors  = 2
n_q_groups   = 1
print_time   = 0.05
print_q_time = 0.05
mol_density  = 1.0E-3
dr           = 1.0
dt           = 0.01
dt_q         = 0.02 # The time step of the quantum dynamics must at least double the time step of Mxll.

create.inp_file(mxll_dimensions=dim, mxll_boundaries=boundaries, mxll_box_size=box_size, 
                mxll_total_time=total_time, mxll_dr=dr, mxll_n_detectors=n_detectors, 
                mxll_dt_det_print=print_time, mxll_dt=dt, mxll_dt_q=dt_q, mxll_dt_q_print=print_q_time, 
                mxll_n_q_groups=n_q_groups, mxll_density_factor=mol_density)

Now we have to create the file `mol_group_xxxxxxx.in` where we indicate the number of molecules we want to simulate, their position, from where they will get their input data and whether their information will be printed or not:

In [ ]:
group_id    = 1
n_q_systems = 1
print_option = "print_all"
molecule_file_list = np.full(n_q_systems, 1) # All the molecules get the information from molecule_0000001.in
coordinates_list = np.array([0.0])

create.mol_group_file(file_number=group_id, number_of_q_systems=n_q_systems, print_options=print_option,
                      mol_file_list=molecule_file_list, coordinates=coordinates_list)

We then move all the relevant files to the work directory including the `N-N.skf` file and run the code:

In [ ]:
! cp inp sources.in detectors.in molecule_0000001.in mol_group_0000001.in N-N.skf box_with_1mol/.

In [ ]:
! export OMP_NUM_THREADS=4 && cd box_with_1mol && ../../../build/bin/OMxRTA.e

If we plot again the transmitted power we will not see much difference resepect to the last plot.

In [ ]:
process.plot_trans_spectrum_1D(directory="box_with_1mol")

To get the absorption spectra we need to do the following calculation:

$P(\omega) = -\mathrm{Re}\left\{ (E_x^*(\omega)- E_{0,x}^*(\omega)) \, (H_y(\omega) - H_{0,y}(\omega)) \right\}$

This is done automatically by the following function from `tools`:

In [ ]:
process.plot_abs_spectrum_1D(only_pulse_dir="empty_box", sample_dir="box_with_1mol")

## Build the cavity

Knowing the excitation wavelength, we can create a cavity that resonates with our molecule. For this, we will use two mirrors simulated using the Drude model:

$\frac{\partial^2 P_{x}(t)}{\partial t^2} + \gamma \frac{\partial P_x(t)}{\partial t} = \varepsilon_0 \Omega_p^2 E_x(t)$

Since there is no existing metal capable of capturing such a short wavelength, we will set Drude parameters that are convenient for us. These are $\Omega_p = 34.0 \, \rm eV$ and $\gamma = 0.18092 \, \rm eV$.

We now prepare our new work directory. Let's call it `empty_cavity`.

In [ ]:
! mkdir empty_cavity

We create a new input file.

In [ ]:
dim         = 1
boundaries  = ["cpml"]
box_size    = [2000.0]
total_time  = 300.0
n_detectors = 2
n_media     = 2
print_time  = 0.05
dr          = 1.0
dt          = 0.01

create.inp_file(mxll_dimensions=dim, mxll_boundaries=boundaries, mxll_box_size=box_size, 
                mxll_total_time=total_time, mxll_dr=dr, mxll_n_detectors=n_detectors, 
                mxll_n_media=n_media, mxll_dt_det_print=print_time, mxll_dt=dt)

We also need to create the files `medium_xxxx.in` where we place the coordinates of the two mirrors we will use. We have to keep in mind that this coordinates have to be consistent with the grid dimensions and grid spacing.

In [ ]:
Omega = 34.0 
gamma = 0.18092
dr = 1.0
mirror1_coordinates = np.arange(-80.0, -60.0+dr, dr)
mirror2_coordinates = np.arange(60.0, 80.0+dr, dr)

create.medium_file(file_number=1, medium_type="drude", omega=Omega, gamma=gamma, coordinates=mirror1_coordinates)
create.medium_file(file_number=2, medium_type="drude", omega=Omega, gamma=gamma, coordinates=mirror2_coordinates)

We move again the input files into the work directory.

In [ ]:
! cp inp sources.in detectors.in medium_0001.in medium_0002.in empty_cavity/.

Now we repeat the running in the working directory and then we plot the transmission spectrum:

In [ ]:
! cd empty_cavity && ../../../build/bin/OMxRTA.e

In [ ]:
process.plot_trans_spectrum_1D(directory="empty_cavity")

The first result doesn't give peaks close to 13.9 eV, so now we have to adjust the separation between the mirrors set in `mxll_media_center`.

Repeat the calculation until you get a cavity mode that resonates with the electronic transition of the $N_2$.

## Simulate system cavity+molecule

I think that now you can put all the pieces together and run the last test...